### Import the libraries

In [2]:
import spacy
from spacy import tokenizer
import unicodedata
from bs4 import BeautifulSoup
import nltk
import string
import re
from nltk.stem import WordNetLemmatizer
from nltk.tokenize.toktok import ToktokTokenizer
import CONTRACTION_MAP
import pandas as pd


### Load the covid19 tweet data

In [3]:
rawData =pd.read_excel("COVID_19_vaccine_100.xlsx")
rawData.columns=['tweets']
rawData.head(5)

,tweets
0,@johensley @DarcyShepherd13 @JeffreyGuterman @...
1,May i remind you that the vaccine isnt just fo...
2,"@MJAckermanMDPhD\nHi, Dr. Ackerman. I will lik..."
3,"Shandro, Hinshaw to give COVID-19 vaccine upda..."
4,You ever Noticed This? They just announced Yes...


## Text Processing

#### Removing html tags

<span style="color:white">Often, unstructured text contains a lot of noise, especially if you use techniques like web or screen scraping. HTML tags are typically one of these components which don’t add much value towards understanding and analyzing text.</span>

In [4]:
def strip_html_tags(text):
    soup = BeautifulSoup(text, "html.parser")
    stripped_text = soup.get_text()
    return stripped_text
strip_html_tags('<html><h2>May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO</h2></html>')

'May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO'

#### Removing accented characters
<span style="color:white">Usually in any text corpus, you might be dealing with accented characters/letters, especially if you only want to analyze the English language. Hence, we need to make sure that these characters are converted and standardized into ASCII characters. A simple example — converting é to e.</span>

In [5]:
def remove_accented_chars(text):
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    return text

remove_accented_chars('Sómě Áccěntěd těxt')
remove_accented_chars("May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO")

'May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO'

#### Expanding Contractions
<span style="color:white">Contractions are shortened version of words or syllables. They often exist in either written or spoken forms in the English language. These shortened versions or contractions of words are created by removing specific letters and sounds. In case of English contractions, they are often created by removing one of the vowels from the word. Examples would be, do not to don’t and I would to I’d. Converting each contraction to its expanded, original form helps with text standardization.</span>

In [6]:
def expand_contractions(text, contraction_mapping=CONTRACTION_MAP.CONTRACTION_MAP):
    modified_text=[]
    for words in text.split(' '):
        val = CONTRACTION_MAP.CONTRACTION_MAP.get(words)
        if val is not None:
            modified_text.append(val)
        else:
            modified_text.append(words)
    modified_text=' '.join(modified_text)
    return modified_text

expand_contractions("Y'all are enjoying this class we'd think. It is so cool isnt it?")

"Y'all are enjoying this class we would think. It is so cool is not it?"

#### Removing Special Characters
<span style="color:white">Special characters and symbols are usually non-alphanumeric characters or even occasionally numeric characters (depending on the problem), which add to the extra noise in unstructured text. Usually, simple regular expressions (regexes) can be used to remove them.</span>

In [7]:
def remove_special_characters(text, remove_digits=True):
    text = ''.join([word.lower() for word in text if word not in string.punctuation])
    text = ''.join([re.sub(r'\d+','',word) for word in text if remove_digits])
    return text

remove_special_characters("Well this was fun! What do you think? 123#@!\\//4", remove_digits=True)
#remove_special_characters("May i remind you that the vaccine isnt just for those who @caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO")

'well this was fun what do you think '

#### Removing Stopwords
<span style="color:white">Words which have little or no significance, especially when constructing meaningful features from text, are known as stopwords or stop words. These are usually words that end up having the maximum frequency if you do a simple term or word frequency in a corpus. Typically, these can be articles, conjunctions, prepositions and so on. Some examples of stopwords are a, an, the, and the like.</span>

In [8]:

stopword_list = nltk.corpus.stopwords.words('english')
tokenizer = ToktokTokenizer()
def remove_stopwords(text, is_lower_case=False):
    tokens = tokenizer.tokenize(text)
    tokens = [token.strip() for token in tokens]
    if is_lower_case:
        filtered_tokens = [token for token in tokens if token not in stopword_list]
    else:
        filtered_tokens = [token for token in tokens if token.lower() not in stopword_list]
    filtered_text = ' '.join(filtered_tokens)
    return filtered_text

remove_stopwords("Let us see if we can or can not remove against the stopwords from a sentence.")
remove_stopwords("May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO")

'May remind vaccine isnt caught covid-19 , also prevent ................ https://t.co/htFc4jP4CO'

#### Remomving URLS
<span style="color:white">Words which contains urls are mostly not required for the data analysis</span>

In [9]:
def url_removal(text):
    modified_text = ' '.join([words for words in text.split(' ') if words[0:4]!="http"])
    return modified_text

url_removal("May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO" )


'May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................'

### Lemmatization

In [10]:
wl =WordNetLemmatizer()
def lemmatize_text(text):
    text = ' '.join([wl.lemmatize(i) for i in text.split(' ')])
    return text


### Stemming

In [11]:
ps = nltk.PorterStemmer()
def stem_text(text):
    text = ' '.join([ps.stem(i) for i in text.split(' ')])
    return text

In [12]:
def normalize_corpus(doc, html_stripping=True, contraction_expansion=True,
                     accented_char_removal=True, text_lower_case=True, special_char_removal=True,
                     stopword_removal=True,remove_url=True, lemmatize=True,stem=True,remove_digits=True):

    normalized_corpus = []
    if html_stripping:
        doc= strip_html_tags(doc)
    # remove accented characters
    if accented_char_removal:
        doc = remove_accented_chars(doc)
    # expand contractions
    if contraction_expansion:
        doc = expand_contractions(doc)
    # lowercase the text
    if text_lower_case:
        doc = doc.lower()
    # remove extra newlines
    doc = re.sub(r'[\r|\n|\r\n]+', ' ',doc)
    # remove special characters and\or digits
    if special_char_removal:
        doc=remove_special_characters(doc,remove_digits=True)
     # remove extra whitespace
    doc = re.sub(' +', ' ', doc)
    # remove stopwords
    if stopword_removal:
        doc = remove_stopwords(doc, is_lower_case=text_lower_case)
    if remove_url:
        doc=url_removal(doc)
    #lemmatize text
    if lemmatize:
        doc = lemmatize_text(doc)
    #stemming text
    if stem:
        doc = stem_text(doc)
    normalized_corpus.append(doc)
    return ' '.join(normalized_corpus)

rawData['tweets_cleaned']= rawData['tweets'].apply(lambda x:normalize_corpus(x))
rawData

C:\Users\ragha\AppData\Local\Temp\ipykernel_2760\172976477.py:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


,tweets,tweets_cleaned
0,@johensley @DarcyShepherd13 @JeffreyGuterman @...,johensley darcyshepherd jeffreyguterman realdo...
1,May i remind you that the vaccine isnt just fo...,may remind vaccin caught covid also prevent
2,"@MJAckermanMDPhD\nHi, Dr. Ackerman. I will lik...",mjackermanmdphd hi dr ackerman like know go di...
3,"Shandro, Hinshaw to give COVID-19 vaccine upda...",shandro hinshaw give covid vaccin updat noon
4,You ever Noticed This? They just announced Yes...,ever notic announc yesterday decemb th great v...
...,...,...
95,Must viewing: Principles of vaccines programs ...,must view principl vaccin program control covi...
96,COVID-19 vaccine's protection against virus ou...,covid vaccin protect viru outweigh potenti all...
97,I've spent some time today looking into whethe...,spent time today look whether covid vaccin do ...
98,COVID-19 Vaccine Likely Beneficial For Breastf...,covid vaccin like benefici breastf babi questi...


In [13]:
import spacy
nltk.download('averaged_perceptron_tagger')
# Download NLTK Punkt sentence tokenizer
nltk.download('punkt')
nlp = spacy.load("en_core_web_sm")
spacy_pos_tagged=[]
for i in rawData['tweets_cleaned']:
    sentence_nlp = nlp(i)
    spacy_pos_tagged_sentence = [(word, word.tag_, word.pos_) for word in [i for i in sentence_nlp]]
    spacy_pos_tagged = spacy_pos_tagged+spacy_pos_tagged_sentence

pos_tagged_data = pd.DataFrame(spacy_pos_tagged, columns=['Word', 'POS tag', 'Tag type'])

pos_tagged_data.to_csv('news.csv', index=False, encoding='utf-8')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ragha\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ragha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [14]:
pos_tagged_data


,Word,POS tag,Tag type
0,johensley,NNP,PROPN
1,darcyshepherd,NNP,PROPN
2,jeffreyguterman,NNP,PROPN
3,realdonaldtrump,NN,NOUN
4,differ,VBP,VERB
...,...,...,...
1619,suffer,VBP,VERB
1620,bell,NNP,PROPN
1621,palsi,NNP,PROPN
1622,side,NN,NOUN


In [15]:
pos_tagged_data['Word']=pos_tagged_data['Word'].apply(lambda x:str(x).strip(' '))
pos_tagged_data[pos_tagged_data['Word']=='covid']

,Word,POS tag,Tag type
9,covid,JJ,ADJ
32,covid,JJ,ADJ
44,covid,NN,NOUN
51,covid,JJ,ADJ
64,covid,JJ,ADJ
...,...,...,...
1555,covid,JJ,ADJ
1558,covid,JJ,ADJ
1574,covid,JJ,ADJ
1595,covid,JJ,ADJ


In [16]:
data_count = pos_tagged_data.groupby(["Word","POS tag"]).count()
data_count

Tag type
Word    POS tag          
abort   JJ              1
        NNP             1
absolut NNP             1
access  NN              1
accord  NN              1
...                   ...
ymmv    NN              1
youatmr NN              1
youtub  NN              1
        NNP             1
zu      NNP             1

[969 rows x 1 columns]

In [39]:
pos_word_covid_count = pos_tagged_data[pos_tagged_data['Word']=='covid'].groupby(["Word","POS tag"]).count()
pos_word_covid_count

Tag type
Word  POS tag          
covid JJ             45
      NN             27
      NNP            37
      VB              2

In [41]:
pos_cardinal_count = pos_tagged_data[pos_tagged_data['POS tag']=='CD'].groupby(["Word","POS tag"]).count().sort_values('Tag type',ascending=False)
pos_cardinal_count

,,Tag type
Word,POS tag,
one,CD,5
two,CD,4
million,CD,1
six,CD,1
